# Stage 1 - MAIN generation, seeds 0 and 1 in one session (Kaggle GPU T4 x2, Internet ON)
Expected total (extrapolated from the pilot, not measured at this size): kernel install about 25 min, then about 1.5-2 h per seed. Raw scale 1 for the SAEs (decision of 2026-11-16). Output: `gen_main_seed0.zip`, `gen_main_seed1.zip` (contain harmful model outputs; keep OUT of git). Each stage is resumable; if the session dies, re-run the same cell and it continues.

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

In [ ]:
# Speed kernels for Qwen3.5's linear-attention layers. The pilot ran 114 s per prompt WITHOUT them.
!pip install -q flash-linear-attention
!timeout 1500 pip install -q causal-conv1d --no-build-isolation || echo "causal-conv1d failed or timed out: continue without it"
import importlib
for mod in ("fla", "causal_conv1d"):
    try: importlib.import_module(mod); print(mod, "OK")
    except Exception as e: print(mod, "missing:", type(e).__name__)

In [ ]:
import sys, os, subprocess, glob, time
TAG, NPC, SEEDS = "main", 0, (0, 1)       # main run: all 300 eval prompts, two seeds in one session (pilot used TAG="pilot", NPC=4)
def run(cmd):
    t = time.time(); print(">>", cmd.split("--stage")[-1][:60], flush=True)
    subprocess.run(cmd.replace("python ", "python -u ", 1), shell=True, check=True); print(f"   done in {time.time()-t:.0f}s", flush=True)
def run_seed(SEED):
    OUT = f"runs/{TAG}/seed{SEED}"
    C = f"--sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --out {OUT} --seed {SEED} --n-per-category {NPC}"
    run(f"python scripts/10_run_stage.py --stage original {C}")
    run(f"python scripts/10_run_stage.py --stage plans --dtype float32 --bs 1 {C}")
    run(f"python scripts/10_run_stage.py --stage steer --methods reins random_any random_active harm_only refusal_only {C}")
    run(f"python scripts/10_run_stage.py --stage original --field safe {C}")
    run(f"python scripts/10_run_stage.py --stage plans --field safe --dtype float32 --bs 1 {C}")
    run(f"python scripts/10_run_stage.py --stage steer --field safe --methods reins {C}")
    run(f"python scripts/14_gate_scores.py --sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --n-per-category {NPC} --out runs/{TAG}/gate_scores_seed{SEED}.csv")
    subprocess.run(f"cd runs && zip -qr ../gen_{TAG}_seed{SEED}.zip {TAG}", shell=True)   # intermediate export after each seed
    print(f"seed {SEED} finished", flush=True)

In [ ]:
run_seed(0)

In [ ]:
run_seed(1)

In [ ]:
import json
for SEED in SEEDS:
    print("== seed", SEED)
    for f in sorted(glob.glob(f"runs/{TAG}/seed{SEED}/*.jsonl")):
        rows = [json.loads(l) for l in open(f)]
        if rows and "response" in rows[0]:
            print(os.path.basename(f), "n", len(rows), "empty", sum(not r["response"].strip() for r in rows), "eos", sum(r["eos"] for r in rows), "mean chars", sum(len(r["response"]) for r in rows)//max(1,len(rows)))

In [ ]:
!ls -la gen_*.zip
!rm -rf /tmp/REINS /tmp/sae ~/.cache/huggingface/hub/models--Qwen*